In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from tqdm import tqdm
from scipy.linalg import rq
from scipy.optimize import least_squares
%matplotlib inline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', device)

SAVE_FIGURES = False  # set to True to write the figures used on the slides to ./figures
def savefig(name):
    if SAVE_FIGURES:
        os.makedirs('figures', exist_ok=True)
        plt.savefig(f'figures/{name}.png', dpi=150, bbox_inches='tight')

# Lecture 14 - Camera Models

A pinhole camera in a few lines of numpy: perspective vs. orthographic projection, the dolly zoom, vanishing points with homogeneous coordinates, and camera calibration with the linear (DLT) and non-linear method.

This notebook needs no data files and no GPU.

Run this notebook on Google Colab (see the [readme](https://github.com/chrirupp/cv_course)). Cells that load large pretrained models are faster with a GPU runtime (*Edit > Notebook settings > T4 GPU*).

## A pinhole camera

A 3D point $\mathbf{X}$ (world coordinates) is mapped to the image by $\mathbf{x} \sim K [R \mid \mathbf{t}] \mathbf{X}$: first into camera coordinates ($R$, $\mathbf{t}$, the *extrinsics*), then onto the image plane and into pixels ($K$, the *intrinsics*), and finally we divide by the third (homogeneous) coordinate.

In [ ]:
def intrinsics(f, cx=320, cy=240):
    return np.array([[f, 0, cx], [0, f, cy], [0, 0, 1.0]])

def look_at(camera_center, target, up=(0, 1, 0)):
    """rotation R and translation t of a camera at `camera_center` looking at `target` (x right, y down, z forward)"""
    z = np.asarray(target, float) - camera_center
    z /= np.linalg.norm(z)
    x = np.cross(z, up)
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    R = np.stack([x, y, z])
    return R, -R @ camera_center

def project(X, K, R, t):
    """perspective projection of (N, 3) world points to (N, 2) pixels"""
    x = (K @ (R @ X.T + t[:, None])).T
    return x[:, :2] / x[:, 2:]

def project_orthographic(X, K, R, t, scale):
    """orthographic projection: drop the depth instead of dividing by it"""
    Xc = (R @ X.T + t[:, None]).T
    return Xc[:, :2] * scale + K[:2, 2]

# a small scene: a row of identical houses along a street (y is up in the world)
def house(x0, z0, w=2.0, h=2.0, d=3.0):
    v = np.array([[0, 0, 0], [w, 0, 0], [w, 0, d], [0, 0, d], [0, h, 0], [w, h, 0], [w, h, d], [0, h, d], [w/2, h*1.5, 0], [w/2, h*1.5, d]])
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7),(4,8),(5,8),(7,9),(6,9),(8,9)]
    return v + [x0, 0, z0], edges

houses = [house(-4, z) for z in range(0, 40, 5)] + [house(2, z) for z in range(0, 40, 5)]
X = np.concatenate([v for v, _ in houses])
edges = [(a + 10*i, b + 10*i) for i, (_, e) in enumerate(houses) for a, b in e]

def draw(ax, x, title):
    for a, b in edges:
        ax.plot(x[[a, b], 0], x[[a, b], 1], 'k-', linewidth=1)
    ax.set_xlim(0, 640)
    ax.set_ylim(480, 0)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

C = np.array([0.0, 1.6, -6.0])               # eye height 1.6 m, 6 m in front of the first houses
R, t = look_at(C, [0, 1.6, 10])
K = intrinsics(300)
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
draw(axs[0], project(X, K, R, t), 'Perspective projection')
draw(axs[1], project_orthographic(X, K, R, t, scale=40), 'Orthographic projection')
savefig('camera_perspective_orthographic')
plt.show()

## The dolly zoom

Move the camera away from the scene and zoom in (increase $f$) so that the nearest house keeps its size in the image. The further away the camera, the smaller the *relative* depth differences, and the closer the perspective image gets to an orthographic one. (Hitchcock used this in *Vertigo*.)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

distances = np.geomspace(4, 200, 60)
def dolly_frame(ax, distance):
    C = np.array([0.0, 1.6, -distance])
    R, t = look_at(C, [0, 1.6, 10])
    f = 300 * distance / 6                   # f / distance stays constant: the front houses keep their size
    ax.clear()
    draw(ax, project(X, intrinsics(f), R, t), f'camera {distance:.0f} m away, f = {f:.0f} px')

fig, axs = plt.subplots(1, 4, figsize=(24, 5))
for ax, d in zip(axs, [4, 10, 40, 200]):
    dolly_frame(ax, d)
savefig('camera_dolly_zoom')
plt.show()

fig, ax = plt.subplots(figsize=(7, 5.5))
anim = FuncAnimation(fig, lambda i: dolly_frame(ax, np.concatenate([distances, distances[::-1]])[i]), frames=2 * len(distances), interval=60)
plt.close()
HTML(anim.to_jshtml())

## Vanishing points

Parallel 3D lines with direction $\mathbf{d}$ meet in the image at the vanishing point $\mathbf{v} \sim K R \mathbf{d}$ (the projection of the point at infinity $(\mathbf{d}, 0)$). With homogeneous coordinates (Lecture 5) we can also find it from the image alone: the line through two image points is $\mathbf{l} = \mathbf{x}_1 \times \mathbf{x}_2$, and two lines intersect at $\mathbf{l}_1 \times \mathbf{l}_2$.

In [ ]:
C = np.array([-1.0, 1.6, -6.0])
R, t = look_at(C, [3, 0.5, 10])          # look a bit to the side and down: the street is not parallel to the image plane
K = intrinsics(300)
x = project(X, K, R, t)
hom = lambda p: np.array([p[0], p[1], 1.0])

# the long edges of the houses (along z) and the roof ridges are parallel: direction (0, 0, 1)
parallel = [(a, b) for a, b in edges if np.allclose(X[b] - X[a], [0, 0, 3])]
lines = [np.cross(hom(x[a]), hom(x[b])) for a, b in parallel]
v_from_image = np.cross(lines[0], lines[5])
v_from_image = v_from_image[:2] / v_from_image[2]
v_true = K @ R @ np.array([0, 0, 1.0])
v_true = v_true[:2] / v_true[2]
print('vanishing point from two image lines:', v_from_image.round(2))
print('vanishing point K R d:              ', v_true.round(2))

# vanishing points of all horizontal directions lie on one line: the horizon
d1, d2 = np.array([1.0, 0, 0]), np.array([0, 0, 1.0])
horizon = np.cross(K @ R @ d1, K @ R @ d2)

fig, ax = plt.subplots(figsize=(12, 8))
draw(ax, x, 'Parallel lines meet at the vanishing point; horizontal directions meet on the horizon')
for a, b in parallel:
    ax.plot([x[a, 0], v_true[0]], [x[a, 1], v_true[1]], 'r-', linewidth=0.5, alpha=0.6)
ax.plot(*v_true, 'ro', markersize=10)
xs = np.array([-2000, 3000])
ax.plot(xs, -(horizon[0] * xs + horizon[2]) / horizon[1], 'b--', label='horizon')
ax.set_xlim(min(0, v_true[0] - 50), max(640, v_true[0] + 50))
ax.set_ylim(480, min(0, v_true[1] - 50))
ax.legend()
savefig('camera_vanishing_point')
plt.show()

## Camera calibration

We photograph a calibration object with known 3D points (two checkerboards at a right angle) and detect their 2D projections (here: we simulate them, with 0.5 px of noise; the camera has barrel distortion).

**Linear method (DLT).** Each correspondence $\mathbf{x}_i \sim P \mathbf{X}_i$ gives two linear equations in the 12 entries of $P$. Stack them into $A \mathbf{p} = 0$ and take the right singular vector of the smallest singular value. Then split $P = K[R \mid \mathbf{t}]$: the left $3 \times 3$ block is $KR$, an upper-triangular times an orthogonal matrix, which is exactly an RQ decomposition.

In [ ]:
# calibration object: two 10x8 checkerboards (5 cm squares) on the planes x=0 and z=0
s = 0.05
grid = np.stack(np.meshgrid(np.arange(1, 11), np.arange(1, 9)), -1).reshape(-1, 2) * s
calib_points = np.concatenate([np.stack([np.zeros(len(grid)), grid[:, 1], grid[:, 0]], 1),
                               np.stack([grid[:, 0], grid[:, 1], np.zeros(len(grid))], 1)])

# the true camera, including radial lens distortion (k1 < 0: barrel distortion)
K_true = np.array([[800, 0, 490], [0, 790, 355], [0, 0, 1.0]])   # a 960x720 image
R_true, t_true = look_at(np.array([0.65, 0.5, 0.65]), [0.12, 0.2, 0.12])
k1_true = -0.3

def project_distorted(X, K, R, t, k1):
    Xc = (R @ X.T + t[:, None]).T
    xn = Xc[:, :2] / Xc[:, 2:]                         # normalised image coordinates
    xd = xn * (1 + k1 * (xn ** 2).sum(1, keepdims=True))
    return xd @ K[:2, :2].T + K[:2, 2]

rng = np.random.default_rng(0)
observed = project_distorted(calib_points, K_true, R_true, t_true, k1_true) + rng.normal(0, 0.5, (len(calib_points), 2))

def dlt(X, x):
    A = []
    for (X_, Y_, Z_), (u, v) in zip(X, x):
        P = [X_, Y_, Z_, 1]
        A.append([*P, 0, 0, 0, 0, *(-u * np.array(P))])
        A.append([0, 0, 0, 0, *P, *(-v * np.array(P))])
    _, _, Vt = np.linalg.svd(np.array(A))
    return Vt[-1].reshape(3, 4)

def decompose(P):
    K, R = rq(P[:, :3])
    S = np.diag(np.sign(np.diag(K)))                 # make the diagonal of K positive
    K, R = K @ S, S @ R
    t = np.linalg.solve(K, P[:, 3])
    if np.linalg.det(R) < 0:                          # P is only defined up to sign
        R, t = -R, -t
    return K / K[2, 2], R, t

def reprojection_error(K, R, t, k1=0.0):
    # root mean square distance between reprojected and observed points (px)
    return np.sqrt(((project_distorted(calib_points, K, R, t, k1) - observed) ** 2).sum(1).mean())

K_lin, R_lin, t_lin = decompose(dlt(calib_points, observed))
print('true K:\n', K_true.round(1))
print('linear (DLT) K:\n', K_lin.round(1))
print(f'DLT reprojection error (RMS): {reprojection_error(K_lin, R_lin, t_lin):.2f} px')

**Non-linear method.** The DLT minimises an algebraic error and cannot model lens distortion. We refine all parameters (focal lengths, principal point, rotation, translation and the distortion coefficient $k_1$) by minimising the *reprojection error* directly, starting from the DLT solution.

In [ ]:
from scipy.spatial.transform import Rotation

def unpack(params):
    fx, fy, cx, cy, k1 = params[:5]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    return K, Rotation.from_rotvec(params[5:8]).as_matrix(), params[8:11], k1

def residuals(params, model_distortion):
    K, R, t, k1 = unpack(params)
    return (project_distorted(calib_points, K, R, t, k1 if model_distortion else 0.0) - observed).ravel()

x0 = np.concatenate([[K_lin[0, 0], K_lin[1, 1], K_lin[0, 2], K_lin[1, 2], 0.0], Rotation.from_matrix(R_lin).as_rotvec(), t_lin])
results = {'linear (DLT)': (K_lin, R_lin, t_lin, 0.0)}
for name, distortion in [('non-linear, no distortion', False), ('non-linear + radial distortion', True)]:
    K, R, t, k1 = unpack(least_squares(residuals, x0, args=(distortion,)).x)
    results[name] = (K, R, t, k1 if distortion else 0.0)

print(f'{"":32s}  fx     fy     cx     cy     k1     RMS error (px)')
print(f'{"true":32s} {K_true[0,0]:6.1f} {K_true[1,1]:6.1f} {K_true[0,2]:6.1f} {K_true[1,2]:6.1f} {k1_true:6.3f}')
for name, (K, R, t, k1) in results.items():
    print(f'{name:32s} {K[0,0]:6.1f} {K[1,1]:6.1f} {K[0,2]:6.1f} {K[1,2]:6.1f} {k1:6.3f}   {reprojection_error(K, R, t, k1):.2f}')

fig, axs = plt.subplots(1, len(results), figsize=(24, 7))
for ax, (name, (K, R, t, k1)) in zip(axs, results.items()):
    predicted = project_distorted(calib_points, K, R, t, k1)
    ax.scatter(*observed.T, s=12, c='k', label='observed')
    ax.scatter(*predicted.T, s=40, facecolors='none', edgecolors='r', label='reprojected')
    err = predicted - observed
    ax.quiver(*observed.T, *(10 * err).T, color='r', angles='xy', scale_units='xy', scale=1, width=0.003)
    ax.set_title(f'{name}: RMS {reprojection_error(K, R, t, k1):.2f} px (errors x10)')
    ax.set_xlim(0, 960)
    ax.set_ylim(720, 0)
    ax.set_aspect('equal')
    ax.legend(loc='lower left')
savefig('camera_calibration')
plt.show()